In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/RedNeuronal_Colab"
!head -1 profiles_unificado.csv
!head -1 friendships_unificado.csv

/content/drive/MyDrive/Colab Notebooks/RedNeuronal_Colab
id,username,pixel_avatar,interests,activity_score,is_online,is_moderator,is_public,tags_agregados
id,requester_id,receiver_id,is_mutual


In [ ]:
import os

print("features.py (debe ser True):", os.path.isfile("scripts/features.py"))
print("train_model.py (debe ser False):", os.path.isfile("scripts/train_model.py"))
print("sintetico_features.csv (debe ser True):", os.path.isfile("sintetico_features.csv"))

features.py (debe ser True): True
train_model.py (debe ser False): False
sintetico_features.csv (debe ser True): True


In [ ]:
!pip install -r requirements.txt

In [ ]:
!cat scripts/train_neural_net.py

# scripts/train_neural_net.py
"""
Entrena CompatibilityNet con el split guardado en data/splits.npz.

Correr desde la raiz del proyecto, con el venv activado:
    python scripts/train_neural_net.py

La funcion entrenar() tambien es usada por scripts/feedback_loop.py
para reentrenar con hiperparametros ajustados si la evaluacion falla.
"""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from app.ml.model import CompatibilityNet

DATA_PATH = Path("data/splits.npz")
MODEL_PATH = Path("models/compatibility_net.pt")
BATCH_SIZE = 16
MAX_EPOCHS = 200
PATIENCE = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4


def cargar_split() -> dict[str, np.ndarray]:
    data = np.load(DATA_PATH)
    return {k: data[k] for k in data.files if k != "feature_names"}


def a_tensor_dataset(X: np.ndarray, y: np.ndarray) -> TensorDataset:
    X_t = tor

In [ ]:
!cat app/ml/model.py

# app/ml/model.py
"""
Red neuronal para el predictor de compatibilidad de amistad.

Arquitectura: MLP simple (dataset chico de 94 ejemplos)
    Linear(5->8) + ReLU + Dropout(0.3)
    Linear(8->4) + ReLU + Dropout(0.2)
    Linear(4->1) + Sigmoid
"""

import torch
import torch.nn as nn


class CompatibilityNet(nn.Module):
    def __init__(self, input_dim: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(4, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
!pwd
!ls scripts
!ls app

/content/drive/MyDrive/Colab Notebooks/RedNeuronal_Colab
colab_busqueda_arquitectura.py	feedback_loop.py  split_dataset_csv.py
evaluate_model.py		__init__.py	  train_neural_net.py
features.py			__pycache__
config.py    database.py  main.py  models	repositories  views
controllers  __init__.py  ml	   __pycache__	services


In [ ]:
!python scripts/split_dataset_csv.py

Dataset total: 8000 ejemplos (4000 positivos, 4000 negativos)
Train: 5600 ejemplos (2800 positivos)
Val:   1200 ejemplos (600 positivos)
Test:  1200 ejemplos (600 positivos)

Split guardado en: /content/drive/MyDrive/Colab Notebooks/RedNeuronal_Colab/data/splits.npz


In [ ]:
!python scripts/train_neural_net.py

Epoca   1 | train_loss: 0.7625 | val_loss: 0.6942
Epoca   2 | train_loss: 0.7027 | val_loss: 0.6941
Epoca   3 | train_loss: 0.6983 | val_loss: 0.6936
Epoca   4 | train_loss: 0.6943 | val_loss: 0.6925
Epoca   5 | train_loss: 0.6930 | val_loss: 0.6916
Epoca   6 | train_loss: 0.6897 | val_loss: 0.6868
Epoca   7 | train_loss: 0.6856 | val_loss: 0.6773
Epoca   8 | train_loss: 0.6763 | val_loss: 0.6559
Epoca   9 | train_loss: 0.6541 | val_loss: 0.6179
Epoca  10 | train_loss: 0.6216 | val_loss: 0.5819
Epoca  11 | train_loss: 0.5819 | val_loss: 0.5244
Epoca  12 | train_loss: 0.5640 | val_loss: 0.5207
Epoca  13 | train_loss: 0.5566 | val_loss: 0.4998
Epoca  14 | train_loss: 0.5445 | val_loss: 0.4937
Epoca  15 | train_loss: 0.5412 | val_loss: 0.4932
Epoca  16 | train_loss: 0.5394 | val_loss: 0.4879
Epoca  17 | train_loss: 0.5320 | val_loss: 0.4880
Epoca  18 | train_loss: 0.5496 | val_loss: 0.5030
Epoca  19 | train_loss: 0.5370 | val_loss: 0.4905
Epoca  20 | train_loss: 0.5314 | val_loss: 0.4871


In [ ]:
!python scripts/evaluate_model.py

Ejemplos de test: 1200 (600 positivos)

--- Evaluacion con distintos umbrales ---
Umbral 0.50 -> Precision: 0.83 | Recall: 0.77 | F1: 0.80
Umbral 0.40 -> Precision: 0.83 | Recall: 0.77 | F1: 0.80
Umbral 0.35 -> Precision: 0.83 | Recall: 0.77 | F1: 0.80
Umbral 0.30 -> Precision: 0.83 | Recall: 0.77 | F1: 0.80
Umbral 0.25 -> Precision: 0.83 | Recall: 0.77 | F1: 0.80

--- Reporte completo con umbral elegido (0.5) ---
               precision    recall  f1-score   support

no_compatible       0.79      0.84      0.81       600
   compatible       0.83      0.77      0.80       600

     accuracy                           0.81      1200
    macro avg       0.81      0.81      0.81      1200
 weighted avg       0.81      0.81      0.81      1200



In [ ]:
%run scripts/colab_busqueda_arquitectura.py

      arquitectura  dropout  AUC media  AUC desvio  accuracy media  F1 media
            16-8-4   0.0000     0.8424      0.0097          0.8161    0.8115
                 8   0.2000     0.8421      0.0095          0.8160    0.8114
LogisticRegression      NaN     0.8421      0.0096          0.8160    0.8114
                 8   0.0000     0.8420      0.0093          0.8160    0.8114
              16-8   0.2000     0.8416      0.0091          0.8160    0.8114
             32-16   0.2000     0.8412      0.0092          0.8161    0.8115
               8-4   0.0000     0.8410      0.0084          0.8160    0.8114
               8-4   0.2000     0.8409      0.0083          0.8160    0.8114
              16-8   0.0000     0.8405      0.0089          0.8160    0.8114
            16-8-4   0.2000     0.8403      0.0098          0.8161    0.8115
             32-16   0.0000     0.8399      0.0092          0.8160    0.8114

Resultados guardados en: /content/drive/MyDrive/Colab Notebooks/RedNeuronal

In [ ]:
import os
from google.colab import files

archivos = ["models/compatibility_net.pt", "data/splits.npz"]

for ruta in archivos:
    if os.path.isfile(ruta):
        print(f"OK  {ruta}  ({os.path.getsize(ruta)} bytes)")
        files.download(ruta)
    else:
        print(f"NO EXISTE: {ruta}")

OK  models/compatibility_net.pt  (3533 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OK  data/splits.npz  (386204 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>